In [0]:
import os
import sys
from databricks.sdk.runtime import dbutils

notebook_path = "/Workspace" + os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().replace("/notebooks", "")
    # "/AI/creri/creri_model/files/"
)

sys.path.append(notebook_path)
os.chdir(notebook_path)
print(notebook_path)

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from datetime import datetime
import pandas as pd
from pandas import DataFrame as pandasDataFrame
from pyspark.sql import DataFrame
from dateutil.relativedelta import relativedelta
from typing import Dict
import json
import ast
import logging
import hashlib
from src.data_preparation.features_engineering import features_engineering, get_version

In [0]:
%run ./materialization_utils/materialization_utils

In [0]:
cod_layerName = dbutils.widgets.get("cod_layerName")
snapshot_date = dbutils.widgets.get("snapshot_date")
RunId = dbutils.widgets.get("RunId")
cod_pipelineID = dbutils.widgets.get("cod_pipelineID")
cod_productName = dbutils.widgets.get("cod_productName")
env = dbutils.widgets.get("env")
flg_lastSnap = 1
version = get_version()
iid_processID = 1

general_utils = GeneralUtils(spark)
metric_materializer = MetricMaterializer(spark)
logger = general_utils.set_logging()

config = general_utils.load_config(general_utils.get_variable(cod_productName)['config_setup_path'])
metadata_path = general_utils.get_variable(cod_productName)['metadata_path']

config_data = general_utils.load_config(general_utils.get_variable(cod_productName)['config_data_path'])
encode_maps = {"SDF_CANLE_ATTIVAZ_ATT_CLIIN": config_data['input_features']["MetricaSdfAttivazioneAttivi"]["SDF_CANLE_ATTIVAZ_ATT_CLIIN"]['encoding'],
              "MACROSEGMENTO": config_data['input_features']["CustomBase"]["MACROSEGMENTO"]['encoding'],
              "MACROSEGMENTO_CLIIN": config_data['input_features']["CustomBase"]["MACROSEGMENTO_CLIIN"]['encoding'] } 
 
catalog_name = config['environment'][env]['catalog']
schema_name = config['environment'][env]['schema']

metriche_input_params = [x for x in config['master_training_final'] if cod_layerName in x['type']][0]
metriche_input_params

table_name = metriche_input_params['name']

In [0]:
if cod_productName == "creri":
    if cod_layerName == "INFER_SERVING":
        print(snapshot_date)
    else:  
        snapshot_date = general_utils.date_to_start_month(snapshot_date)
        try:
            snapshot_date = general_utils.replace_month_date(snapshot_date, int(metriche_input_params['today_minus_n_months']))
        except:
            snapshot_date = general_utils.replace_month_date(snapshot_date, 0)
        print(snapshot_date)

In [0]:
ih_mstr_volturante = catalog_name+"."+schema_name+"."+[x['name'] for x in config['master_training'] if x['type'] == "PREP_MASTER_TRAINING" and 'volturante' in x['name']][0]
ih_mstr_volturato = catalog_name+"."+schema_name+"."+[x['name'] for x in config['master_training'] if x['type'] == "PREP_MASTER_TRAINING" and 'volturato' in x['name']][0]
master_final_path = catalog_name+"."+schema_name+"."+table_name

#### if data snapshot overwrite is required, delete the exist snapshot data ####
snapshot_overwrite_res = metric_materializer.check_snapshot_overwrite(catalog_name, 
                         schema_name, 
                         table_name, 
                         snapshot_date, 
                         config['master_training_final'][0]['snapshot_overwrite'])

#### check if metric is updated on the latest version ####
try:
    is_metrics_updated = metric_materializer.is_metrics_updated(catalog_name,
                                           schema_name, 
                                           table_name, 
                                           get_version(), 
                                           snapshot_date)
    logger.info(f"Updating metrics on new version: {is_metrics_updated}")
except Exception as e:
    # logger.info(f"Writing new snapshot on: {snapshot_date}")
    logger.info("Error metric updating:" + str(e))
    raise

metrics_used_list = metric_materializer.get_metrics_used(config, metriche_input_params)
hash_data_version = metric_materializer.get_met_version(catalog_name, schema_name, metrics_used_list, metriche_input_params['class_name'], version, snapshot_date)

#### if metric is not updated -> select the snapshot date not. updated ####
if not is_metrics_updated:
    logger.info(f"Identified a new version of the metric, materializing with the selected snapshot date")
    data_version_collection = (
        spark.read.table(f"{catalog_name}.{schema_name}.{table_name}") \
        .where(f"flg_lastSnap = 1")\
        .select("snapshot_date", "version") \
        .orderBy("snapshot_date", F.col("version").desc()) \
        .dropDuplicates(["snapshot_date"]) \
        .toLocalIterator()
    )

    #### loop on the snapshot date not updated and materialize them ####
    for data_object in data_version_collection:
        if data_object["version"] != hash_data_version:

            #### set to 0 the old data need to be updated (per snapshot date) ####
            metric_versioning = metric_materializer.metric_versioning(catalog_name, schema_name, table_name, data_object['snapshot_date'], metriche_input_params['historicization'])
            logger.info(f"{metric_versioning}")
            logger.info(f"  Updating snapshot_date: snapshot_date: {data_object['snapshot_date']}")

            #### materialize the new data on the selected snapshot date ####
            logger.info(f"  Updating snapshot_date: Reading master: {ih_mstr_volturante}")
            master_volturante = general_utils.master_reader(ih_mstr_volturante, data_object['snapshot_date'])
            logger.info(f"  Updating snapshot_date: Reading master: {ih_mstr_volturato}")
            master_volturato = general_utils.master_reader(ih_mstr_volturato, data_object['snapshot_date'])
            logger.info(f"  Updating snapshot_date: Removing tech columns from master: {ih_mstr_volturante}")

            master_volturante_clean = metric_materializer.remove_tech_columns(master_volturante)
            logger.info(f"  Updating snapshot_date: Removing tech columns from master: {ih_mstr_volturato}")
            master_volturato_clean = metric_materializer.remove_tech_columns(master_volturato)

            target_cols = metric_materializer.remove_tech_columns(spark.read.table(f"{catalog_name}.{schema_name}.creri_met_targetcontofattclientimorosi")).columns
            logger.info("  target_cols ", str([x for x in target_cols if "SDF_CANLE_ATTIVAZ_ATT" in x]))
            cols_to_rename = list(set(master_volturante_clean.columns) - set(target_cols))
            logger.info("  cols_to_rename ", str([x for x in cols_to_rename if "SDF_CANLE_ATTIVAZ_ATT" in x]))
            for c in cols_to_rename + ["PK_CONTO_FATT"]:
                if c not in ["PK_PROPOSAL_LINE_ITEM"]:
                    master_volturante_clean = master_volturante_clean.withColumnRenamed(c, c + "_CLIIN")

            logger.info(f"  Updating snapshot_date: Joining master tables on PK_PROPOSAL_LINE_ITEM column")
            master_df_final = master_volturante_clean.join(master_volturato_clean, on="PK_PROPOSAL_LINE_ITEM", how="inner")\
                .drop("DATA_VOLTURA_CLIIN", "DATA_FIRMA_VOLTURA_CLIIN")
            logger.info(f"  Updating snapshot_date: Applying dataset splitting rules on master table")

            result_df_splitting = master_df_final.withColumn("SDF_CANLE_ATTIVAZ_ATT_CLIIN",
                                                                F.when(F.col("SDF_CANLE_ATTIVAZ_ATT_CLIIN").isNull(), "")\
                                                                    .otherwise(F.col("SDF_CANLE_ATTIVAZ_ATT_CLIIN")))\
                                                    .withColumn("MACROSEGMENTO",
                                                                F.when(F.col("MACROSEGMENTO").isNull(), "")\
                                                                    .otherwise(F.col("MACROSEGMENTO")))\
                                                    .withColumn("MACROSEGMENTO_CLIIN",
                                                                F.when(F.col("MACROSEGMENTO_CLIIN").isNull(), "")\
                                                                    .otherwise(F.col("MACROSEGMENTO_CLIIN")))
            logger.info(f"  Updating snapshot_date: Applying feature engineering on master table")

            #### apply feature engineering on the master table ####
            result_df_engineered = features_engineering(result_df_splitting, "training_master_creation", encode_maps)

            #### drop excessive columns ####
            cols_to_drop = ["PK_CONTO_FATT", "PK_PDF", "PK_ACCOUNT", "PK_ASSET_LINE_ITEM", "PK_QUOTE_PROPOSAL", "PK_ORDER_LINE_ITEM"]
            for c in ["", "_CLIIN"]:
                cols_to_drop = [x + c for x in cols_to_drop]
                result_df_engineered = result_df_engineered.drop(*cols_to_drop)

            #### cast fields based on the configuration metadata (schema enforcement) ####
            logger.info("  Updating snapshot_date: Casting fields based on configuration data")
            schema = metric_materializer.remove_tech_columns(spark.read.table(catalog_name + "." + schema_name + "." + table_name)).schema
            schema_map = {field.name: field.dataType for field in schema.fields}
            common_cols, df_only_cols = metric_materializer.get_schema_fields(schema_map, result_df_engineered)
            result_df_engineered = metric_materializer.cast_columns(common_cols, schema_map, result_df_engineered)

            #### adding technical columns ####
            logger.info(f"  Updating snapshot_date: Adding tech columns")
            result_df_final = metric_materializer.add_tech_columns(
                result_df_engineered, 
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"), 
                RunId, 
                hash_data_version, 
                flg_lastSnap,
                metriche_input_params['join_key'], 
                data_object['snapshot_date'])

            result_df_final = result_df_final.withColumn("snapshot_date", F.to_date(F.col("snapshot_date"), "yyyy-MM-dd"))

            #### write data on the target table ####
            logger.info(f"  Updating snapshot_date: Writing tables")
            general_utils.write_table(
                catalog_name, 
                schema_name, 
                table_name, 
                result_df_final, 
                "append")

            #### populate technical column end_date closing the write operation ####
            logger.info(f"  Updating snapshot_date: Populating materialization end date column")
            metric_materializer.populate_metrics_end_date(
                    catalog_name, 
                    database=schema_name, 
                    table=table_name,
                    snapshot_date=data_object['snapshot_date'])
            logger.info(f"  Updating snapshot_date: MASTER WRITTEN ON: {catalog_name}.{schema_name}.{table_name}")
        else:
            logger.info(f"  Updating snapshot_date: Materialized data updated for snapshot_date = {snapshot_date}")
else:
    logger.info(f"Materialized data updated!!")

#### check if the snapshot date is already materialized ####
if metric_materializer.check_snapshot_data(f"{catalog_name}.{schema_name}.{table_name}", snapshot_date):
    logger.info(f"DF with the selected snapshot date already exists!!")
else:
    #### set to 0 the old data need to be updated (per snapshot date) ####
    metric_versioning = metric_materializer.metric_versioning(catalog_name, schema_name, table_name, snapshot_date, metriche_input_params['historicization'])
    logger.info(f"{metric_versioning}")

    #### materialize the new data on the selected snapshot date ####
    logger.info(f"Reading master: {ih_mstr_volturante}")
    master_volturante = general_utils.master_reader(ih_mstr_volturante, snapshot_date)
    logger.info(f"Reading master: {ih_mstr_volturato}")
    master_volturato = general_utils.master_reader(ih_mstr_volturato, snapshot_date)
    logger.info(f"Removing tech columns from master: {ih_mstr_volturante}")
    master_volturante_clean = metric_materializer.remove_tech_columns(master_volturante)
    logger.info(f"Removing tech columns from master: {ih_mstr_volturato}")
    master_volturato_clean = metric_materializer.remove_tech_columns(master_volturato)
    
    target_cols = metric_materializer.remove_tech_columns(spark.read.table(f"{catalog_name}.{schema_name}.creri_met_targetcontofattclientimorosi")).columns
    print("target_cols ", str([x for x in target_cols if "SDF_CANLE_ATTIVAZ_ATT" in x]))
    cols_to_rename = list(set(master_volturante_clean.columns) - set(target_cols))
    print("cols_to_rename ", str([x for x in cols_to_rename if "SDF_CANLE_ATTIVAZ_ATT" in x]))
    for c in cols_to_rename + ["PK_CONTO_FATT"]:
        if c not in ["PK_PROPOSAL_LINE_ITEM"]:
            master_volturante_clean = master_volturante_clean.withColumnRenamed(c, c + "_CLIIN")

    logger.info(f"Joining master tables on PK_PROPOSAL_LINE_ITEM column")
    master_df_final = master_volturante_clean.join(master_volturato_clean, on="PK_PROPOSAL_LINE_ITEM", how="inner")\
        .drop("DATA_VOLTURA_CLIIN", "DATA_FIRMA_VOLTURA_CLIIN")
    logger.info(f"Applying dataset splitting rules on master table")

    result_df_splitting = master_df_final.withColumn("SDF_CANLE_ATTIVAZ_ATT_CLIIN",
                                                         F.when(F.col("SDF_CANLE_ATTIVAZ_ATT_CLIIN").isNull(), "")\
                                                             .otherwise(F.col("SDF_CANLE_ATTIVAZ_ATT_CLIIN")))\
                                            .withColumn("MACROSEGMENTO",
                                                         F.when(F.col("MACROSEGMENTO").isNull(), "")\
                                                             .otherwise(F.col("MACROSEGMENTO")))\
                                            .withColumn("MACROSEGMENTO_CLIIN",
                                                         F.when(F.col("MACROSEGMENTO_CLIIN").isNull(), "")\
                                                             .otherwise(F.col("MACROSEGMENTO_CLIIN")))
    logger.info(f"Applying feature engineering on master table")

    #### apply feature engineering on master table ####
    result_df_engineered = features_engineering(result_df_splitting, "training_master_creation", encode_maps)


    cols_to_drop = ["PK_CONTO_FATT", "PK_PDF", "PK_ACCOUNT", "PK_ASSET_LINE_ITEM", "PK_QUOTE_PROPOSAL", "PK_ORDER_LINE_ITEM"]
    for c in ["", "_CLIIN"]:
        cols_to_drop = [x + c for x in cols_to_drop]
        result_df_engineered = result_df_engineered.drop(*cols_to_drop)

    #### cast fields based on the configuration metadata (schema enforcement) ####
    logger.info("Casting fields based on configuration data")
    schema = metric_materializer.remove_tech_columns(spark.read.table(catalog_name + "." + schema_name + "." + table_name)).schema
    schema_map = {field.name: field.dataType for field in schema.fields}
    common_cols, df_only_cols = metric_materializer.get_schema_fields(schema_map, result_df_engineered)
    result_df_engineered = metric_materializer.cast_columns(common_cols, schema_map, result_df_engineered)

    #### adding technical columns ####
    logger.info(f"Adding tech columns")
    result_df_final = metric_materializer.add_tech_columns(
        result_df_engineered, 
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"), 
        RunId, 
        hash_data_version, 
        flg_lastSnap,
        metriche_input_params['join_key'], 
        snapshot_date)

    result_df_final = result_df_final.withColumn("snapshot_date", F.to_date(F.col("snapshot_date"), "yyyy-MM-dd"))

    #### write data on the target table ####
    logger.info(f"Writing tables")
    general_utils.write_table(
        catalog_name, 
        schema_name, 
        table_name, 
        result_df_final, 
        "append")

    #### populate technical column end_date closing the write operation ####
    logger.info(f"Populating materialization end date column")
    metric_materializer.populate_metrics_end_date(
            catalog_name, 
            database=schema_name, 
            table=table_name,
            snapshot_date=snapshot_date)
    logger.info(f"MASTER WRITTEN ON: {catalog_name}.{schema_name}.{table_name}")

In [0]:
dbutils.notebook.exit("Success!!")

---